# Statistical and Machine Learning Techniques for Flight Delay Prediction and Carbon Emission Optimisation in European Aviation Networks

**Student:** Damilola Tanimowo  
**Student ID:** 2025261  
**Programme:** MSc Data Analytics  
**Institution:** CCT College Dublin  

In [3]:
# ─────────────────────────────────────────────────────────────────
# SECTION 1: Setup & Library Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import gzip
import os
import warnings
warnings.filterwarnings('ignore')

# ── Plot settings ──────────────────────────────────────────────
plt.rcParams['figure.figsize']   = (14, 5)
plt.rcParams['font.size']        = 12
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False
sns.set_style('whitegrid')

# ── Reproducibility ────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Confirm versions ───────────────────────────────────────────
print("✅ Libraries loaded successfully")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")

✅ Libraries loaded successfully
   pandas  : 2.2.2
   numpy   : 1.26.4


In [5]:
# ─────────────────────────────────────────────────────────────────
# SECTION 2: Data Loading
# ─────────────────────────────────────────────────────────────────
# loading all 12 monthly flight files and combine them into one
# master DataFrame. Each file covers one seasonal sample month
# (March, June, September, December) across 2021, 2022, and 2023.
# EUROCONTROL releases only these four months per year to provide
# seasonal coverage.

flight_files = [
    'Flights_20210301_20210331.csv.gz',
    'Flights_20210601_20210630.csv.gz',
    'Flights_20210901_20210930.csv.gz',
    'Flights_20211201_20211231.csv.gz',
    'Flights_20220301_20220331.csv.gz',
    'Flights_20220601_20220630.csv.gz',
    'Flights_20220901_20220930.csv.gz',
    'Flights_20221201_20221231.csv.gz',
    'Flights_20230301_20230331.csv.gz',
    'Flights_20230601_20230630.csv.gz',
    'Flights_20230901_20230930.csv.gz',
    'Flights_20231201_20231231.csv.gz',
]

frames = []

for fname in flight_files:
    with gzip.open(fname, 'rt', encoding='utf-8', errors='replace') as f:
        df_month = pd.read_csv(f)
    frames.append(df_month)
    print(f"   ✅ Loaded {fname}  →  {len(df_month):,} rows")

df = pd.concat(frames, ignore_index=True)

print(f"\n{'─'*55}")
print(f"   TOTAL FLIGHTS LOADED : {len(df):,}")
print(f"   TOTAL COLUMNS        : {len(df.columns)}")
print(f"   COLUMNS              : {list(df.columns)}")

   ✅ Loaded Flights_20210301_20210331.csv.gz  →  250,247 rows
   ✅ Loaded Flights_20210601_20210630.csv.gz  →  447,215 rows
   ✅ Loaded Flights_20210901_20210930.csv.gz  →  652,298 rows
   ✅ Loaded Flights_20211201_20211231.csv.gz  →  570,200 rows
   ✅ Loaded Flights_20220301_20220331.csv.gz  →  576,258 rows
   ✅ Loaded Flights_20220601_20220630.csv.gz  →  813,452 rows
   ✅ Loaded Flights_20220901_20220930.csv.gz  →  826,994 rows
   ✅ Loaded Flights_20221201_20221231.csv.gz  →  646,840 rows
   ✅ Loaded Flights_20230301_20230331.csv.gz  →  681,919 rows
   ✅ Loaded Flights_20230601_20230630.csv.gz  →  884,823 rows
   ✅ Loaded Flights_20230901_20230930.csv.gz  →  899,495 rows
   ✅ Loaded Flights_20231201_20231231.csv.gz  →  697,362 rows

───────────────────────────────────────────────────────
   TOTAL FLIGHTS LOADED : 7,947,103
   TOTAL COLUMNS        : 18
   COLUMNS              : ['ECTRL ID', 'ADEP', 'ADEP Latitude', 'ADEP Longitude', 'ADES', 'ADES Latitude', 'ADES Longitude', 'FILED OF

## Section 3: Data Cleaning & Feature Engineering

The raw EUROCONTROL flight records require several preprocessing steps before modelling can begin. Datetime columns are parsed, delay variables are calculated, and new features are engineered to capture the operational and temporal dynamics known to influence flight delays in European aviation (Baspinar, 2020; Rebollo and Balakrishnan, 2014). A flight is classified as delayed if its arrival delay exceeds 15 minutes, consistent with the industry threshold used by EUROCONTROL and IATA (EUROCONTROL, 2024).

In [10]:
# Parse datetimes
time_cols = ['FILED OFF BLOCK TIME', 'ACTUAL OFF BLOCK TIME',
             'FILED ARRIVAL TIME',   'ACTUAL ARRIVAL TIME']

for col in time_cols:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

# Calculate delays
df['DEP_DELAY_MIN'] = (df['ACTUAL OFF BLOCK TIME'] - df['FILED OFF BLOCK TIME']).dt.total_seconds() / 60
df['ARR_DELAY_MIN'] = (df['ACTUAL ARRIVAL TIME']   - df['FILED ARRIVAL TIME']).dt.total_seconds()   / 60

# Target variables
df['IS_DELAYED']  = (df['ARR_DELAY_MIN'] > 15).astype(int)
df['DISTANCE_KM'] = (df['Actual Distance Flown (nm)'] * 1.852).round(1)

# Temporal features
df['YEAR']        = df['FILED OFF BLOCK TIME'].dt.year
df['MONTH']       = df['FILED OFF BLOCK TIME'].dt.month
df['DAY_OF_WEEK'] = df['FILED OFF BLOCK TIME'].dt.dayofweek
df['HOUR_OF_DAY'] = df['FILED OFF BLOCK TIME'].dt.hour
df['DATE']        = df['FILED OFF BLOCK TIME'].dt.date
df['IS_PEAK_HOUR']= df['HOUR_OF_DAY'].apply(lambda h: 1 if (6<=h<=10) or (16<=h<=20) else 0)
df['IS_WEEKEND']  = (df['DAY_OF_WEEK'] >= 5).astype(int)
df['SEASON']      = df['MONTH'].map({3:'Spring', 6:'Summer', 9:'Autumn', 12:'Winter'})

# Rename columns
df = df.rename(columns={
    'ECTRL ID'                  : 'FLIGHT_ID',
    'ADEP'                      : 'DEP_AIRPORT',
    'ADES'                      : 'ARR_AIRPORT',
    'ADEP Latitude'             : 'DEP_LAT',
    'ADEP Longitude'            : 'DEP_LON',
    'ADES Latitude'             : 'ARR_LAT',
    'ADES Longitude'            : 'ARR_LON',
    'AC Type'                   : 'AIRCRAFT_TYPE',
    'AC Operator'               : 'AIRLINE',
    'ICAO Flight Type'          : 'FLIGHT_TYPE',
    'Actual Distance Flown (nm)': 'DISTANCE_NM',
})

# Drop rows with missing delay values
before = len(df)
df     = df.dropna(subset=['DEP_DELAY_MIN', 'ARR_DELAY_MIN'])
dropped = before - len(df)

print(f"Rows loaded   : {before:,}")
print(f"Rows dropped  : {dropped:,} ({dropped/before*100:.2f}%)")
print(f"Rows remaining: {len(df):,}")
print(f"\nDelay summary:")
print(f"  Avg departure delay : {df['DEP_DELAY_MIN'].mean():.1f} min")
print(f"  Avg arrival delay   : {df['ARR_DELAY_MIN'].mean():.1f} min")
print(f"  Flights delayed >15 : {df['IS_DELAYED'].sum():,} ({df['IS_DELAYED'].mean()*100:.1f}%)")
print(f"\nYears  : {sorted(df['YEAR'].unique())}")
print(f"Seasons: {sorted(df['SEASON'].dropna().unique())}")

Rows loaded   : 7,947,103
Rows dropped  : 0 (0.00%)
Rows remaining: 7,947,103

Delay summary:
  Avg departure delay : 5.1 min
  Avg arrival delay   : 3.9 min
  Flights delayed >15 : 1,423,609 (17.9%)

Years  : [2021, 2022, 2023]
Seasons: ['Autumn', 'Spring', 'Summer', 'Winter']


## Exploratory Data Analysis (EDA)

In [12]:
print(f"Dataset shape        : {df.shape}")
print(f"Date range           : {df['FILED OFF BLOCK TIME'].min().date()} to {df['FILED OFF BLOCK TIME'].max().date()}")
print(f"Total flights        : {len(df):,}")
print(f"Delayed flights (>15): {df['IS_DELAYED'].sum():,} ({df['IS_DELAYED'].mean()*100:.1f}%)")
print(f"On-time flights      : {(df['IS_DELAYED']==0).sum():,} ({(df['IS_DELAYED']==0).mean()*100:.1f}%)")
print(f"Unique aircraft types: {df['AIRCRAFT_TYPE'].nunique():,}")
print(f"Unique airlines      : {df['AIRLINE'].nunique():,}")
print(f"Unique dep airports  : {df['DEP_AIRPORT'].nunique():,}")
print(f"Unique arr airports  : {df['ARR_AIRPORT'].nunique():,}")
print(f"\nFlights per year:")
print(df['YEAR'].value_counts().sort_index().to_string())
print(f"\nFlights per season:")
print(df['SEASON'].value_counts().to_string())
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum()>0].to_string())

Dataset shape        : (7947103, 30)
Date range           : 2021-03-01 to 2023-12-31
Total flights        : 7,947,103
Delayed flights (>15): 1,423,609 (17.9%)
On-time flights      : 6,523,494 (82.1%)
Unique aircraft types: 322
Unique airlines      : 830
Unique dep airports  : 2,259
Unique arr airports  : 2,216

Flights per year:
YEAR
2021    1919960
2022    2863544
2023    3163599

Flights per season:
SEASON
Autumn    2378787
Summer    2145490
Winter    1914402
Spring    1508424

Missing values:
DEP_LAT             2573
DEP_LON             2573
ARR_LAT             3104
ARR_LON             3104
AC Registration    17636
Requested FL        2043


In [15]:
before = len(df)
df = df.dropna(subset=['DEP_LAT', 'DEP_LON'])
print(f"Rows dropped (missing dep coordinates) : {before - len(df):,}")
print(f"Rows remaining                         : {len(df):,}")
print(f"\nRemaining missing values:")
remaining = df.isnull().sum()[df.isnull().sum() > 0]
print(remaining.to_string() if len(remaining) > 0 else "None")

Rows dropped (missing dep coordinates) : 2,573
Rows remaining                         : 7,944,530

Remaining missing values:
ARR_LAT             3104
ARR_LON             3104
AC Registration    17620
Requested FL        2043


### Descriptive Statistics

Before producing any visualisations, a full descriptive statistical summary is generated 
across the key numerical variables. This provides an objective baseline understanding of 
the dataset's central tendency, spread, and distributional properties